# Загрузка датасетов с сайта Zenodo на Kaggle для дальнейшей работы с ними

## Установка библиотек

In [44]:
import subprocess
subprocess.run("pip install kaggle requests", shell=True)

import os, json, shutil

KAGGLE_USER = "daryanikitina"

## Функции

In [45]:
def run(cmd, check=True):
    """Выполняет команду в терминале и печатает вывод"""
    print(f"\n>>> {cmd}")
    env = os.environ.copy()
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
    print(result.stdout)
    print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Код {result.returncode}")
    return result.returncode

def write_meta(folder, slug, title):
    """Создаёт файл dataset-metadata.json, так как Kaggle требует его при любой загрузке"""
    with open(f"{folder}/dataset-metadata.json", "w") as f:
        json.dump({
            "title": title,
            "id": f"{KAGGLE_USER}/{slug}",
            "licenses": [{"name": "CC-BY-4.0"}]
        }, f)

def upload_all_folders(src_folders, slug, title):
    """
    Копирует несколько папок в папку staging и загружает их все одной командой.
    src_folders - список путей к папкам с данными.
    Использует create (не version), потому что создаю датасет с нуля.
    После загрузки удаляет только staging, исходные данные не трогает.
    """
    staging = "C:/kaggle_staging"
    shutil.rmtree(staging, ignore_errors=True)
    os.makedirs(staging, exist_ok=True)

    for src in src_folders:
        folder_name = os.path.basename(src)
        print(f"\nКопируем {folder_name}...")
        shutil.copytree(src, f"{staging}/{folder_name}")
        n = len([f for f in os.listdir(f"{staging}/{folder_name}") if f.endswith(".tif")])
        print(f"  Скопировано: {n} .tif файлов")

    write_meta(staging, slug, title)

    print("\nНачинаем загрузку на Kaggle...")
    run(f"kaggle datasets create -p {staging} --dir-mode tar")

    shutil.rmtree(staging, ignore_errors=True)
    print(f"\nГотово: https://www.kaggle.com/datasets/{KAGGLE_USER}/{slug}")

## Часть 1 - снимки с нефтью (Oil) + маски (Mask_oil): 1200 снимков + 1200 масок

In [46]:
os.makedirs("C:/sar_data/part1", exist_ok=True)

In [47]:
if not os.path.exists("C:/sar_data/part1/Mask_oil"):
    run('wget -c -O C:/sar_data/part1_masks.7z "https://zenodo.org/records/8346860/files/01_Train_Val_Oil_Spill_mask.7z?download=1"')
    run('"C:\\Program Files\\7-Zip\\7z.exe" x C:\\sar_data\\part1_masks.7z -oC:\\sar_data\\part1\\ -y')
    os.remove("C:/sar_data/part1_masks.7z")
else:
    print("Mask_oil уже есть на диске, скачивание пропускаем")

Mask_oil уже есть на диске, скачивание пропускаем


In [48]:
if not os.path.exists("C:/sar_data/part1/Oil"):
    run('wget -c -O C:/sar_data/part1_images.7z "https://zenodo.org/records/8346860/files/01_Train_Val_Oil_Spill_images.7z?download=1"')
    run('"C:\\Program Files\\7-Zip\\7z.exe" x C:\\sar_data\\part1_images.7z -oC:\\sar_data\\part1\\ -y')
    os.remove("C:/sar_data/part1_images.7z")
else:
    print("Oil уже есть на диске, скачивание пропускаем")

Oil уже есть на диске, скачивание пропускаем


*Скачивание датасетов далось кровью и потом, так что первая часть уже была загружена на компьютер в отличие от последующих*

In [49]:
print("\nПроверка части 1:")
for folder in ["Oil", "Mask_oil"]:
    path = f"C:/sar_data/part1/{folder}"
    n = len([f for f in os.listdir(path) if f.endswith(".tif")]) if os.path.exists(path) else "Папка не найдена"
    print(f"  {folder}: {n} .tif файлов")


Проверка части 1:
  Oil: 1200 .tif файлов
  Mask_oil: 1200 .tif файлов


In [50]:
upload_all_folders(
    src_folders=["C:/sar_data/part1/Oil", "C:/sar_data/part1/Mask_oil"],
    slug="sar-oil-spill-part1",
    title="SAR Oil Spill Part1"
)


Копируем Oil...
  Скопировано: 1200 .tif файлов

Копируем Mask_oil...
  Скопировано: 1200 .tif файлов

Начинаем загрузку на Kaggle...

>>> kaggle datasets create -p C:/kaggle_staging --dir-mode tar


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)




Готово: https://www.kaggle.com/datasets/daryanikitina/sar-oil-spill-part1


## Часть 2 - снимки без нефти (No_Oil) + look-alike + маски: 685 снимков No_Oil + 515 Lookalike + маски к ним

In [51]:
os.makedirs("C:/sar_data/part2", exist_ok=True)

In [52]:
if not os.path.exists("C:/sar_data/part2/No_Oil"):
    run('wget -c -O C:/sar_data/part2_nooil.7z "https://zenodo.org/records/8253899/files/01_Train_Val_No_Oil_Images.7z?download=1"')
    run('"C:\\Program Files\\7-Zip\\7z.exe" x C:\\sar_data\\part2_nooil.7z -oC:\\sar_data\\part2\\ -y')
    os.remove("C:/sar_data/part2_nooil.7z")
else:
    print("No_Oil уже есть на диске, скачивание пропускаем")


>>> wget -c -O C:/sar_data/part2_nooil.7z "https://zenodo.org/records/8253899/files/01_Train_Val_No_Oil_Images.7z?download=1"


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)




7-Zip 26.01 (x64) : Copyright (c) 1999-2026 Igor Pavlov : 2026-04-27

Scanning the drive for archives:
1 file, 22931223979 bytes (22 GiB)

Extracting archive: C:\sar_data\part2_nooil.7z
--
Path = C:\sar_data\part2_nooil.7z
Type = 7z
Physical Size = 22931223979
Headers Size = 13859
Method = LZMA:23
Solid = +
Blocks = 1

Everything is Ok

Folders: 1
Files: 685
Size:       28913797780
Compressed: 22931223979




In [54]:
if not os.path.exists("C:/sar_data/part2/Lookalike"):
    run('wget -c -O C:/sar_data/part2_lookalike.7z "https://zenodo.org/records/8253899/files/01_Train_Val_Lookalike_images.7z?download=1"')
    run('"C:\\Program Files\\7-Zip\\7z.exe" x C:\\sar_data\\part2_lookalike.7z -oC:\\sar_data\\part2\\ -y')
    os.remove("C:/sar_data/part2_lookalike.7z")
else:
    print("Lookalike уже есть на диске, скачивание пропускаем")


>>> wget -c -O C:/sar_data/part2_lookalike.7z "https://zenodo.org/records/8253899/files/01_Train_Val_Lookalike_images.7z?download=1"


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)




7-Zip 26.01 (x64) : Copyright (c) 1999-2026 Igor Pavlov : 2026-04-27

Scanning the drive for archives:
1 file, 22993852696 bytes (22 GiB)

Extracting archive: C:\sar_data\part2_lookalike.7z
--
Path = C:\sar_data\part2_lookalike.7z
Type = 7z
Physical Size = 22993852696
Headers Size = 14124
Method = LZMA:23
Solid = +
Blocks = 1

Everything is Ok

Folders: 1
Files: 685
Size:       28904723538
Compressed: 22993852696




In [55]:
if not os.path.exists("C:/sar_data/part2/Mask_No_Oil"):
    run('wget -c -O C:/sar_data/part2_nooil_mask.7z "https://zenodo.org/records/8253899/files/01_Train_Val_No_Oil_mask.7z?download=1"')
    run('"C:\\Program Files\\7-Zip\\7z.exe" x C:\\sar_data\\part2_nooil_mask.7z -oC:\\sar_data\\part2\\ -y')
    os.remove("C:/sar_data/part2_nooil_mask.7z")
else:
    print("Mask_No_Oil уже есть на диске, скачивание пропускаем")


>>> wget -c -O C:/sar_data/part2_nooil_mask.7z "https://zenodo.org/records/8253899/files/01_Train_Val_No_Oil_mask.7z?download=1"

--2026-05-30 02:26:42--  https://zenodo.org/records/8253899/files/01_Train_Val_No_Oil_mask.7z?download=1
Resolving zenodo.org (zenodo.org)... 188.184.103.118, 137.138.153.219, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|188.184.103.118|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 416694 (407K) [application/octet-stream]
Saving to: 'C:/sar_data/part2_nooil_mask.7z'

     0K .......... .......... .......... .......... .......... 12%  554K 1s
    50K .......... .......... .......... .......... .......... 24% 2,24M 0s
   100K .......... .......... .......... .......... .......... 36% 2,27M 0s
   150K .......... .......... .......... .......... .......... 49% 2,35M 0s
   200K .......... .......... .......... .......... .......... 61% 2,40M 0s
   250K .......... .......... .......... .......... .......... 73% 26,4M 0s
  

In [56]:
if not os.path.exists("C:/sar_data/part2/Mask_Lookalike"):
    run('wget -c -O C:/sar_data/part2_lookalike_mask.7z "https://zenodo.org/records/8253899/files/01_Train_Val_Lookalike_mask.7z?download=1"')
    run('"C:\\Program Files\\7-Zip\\7z.exe" x C:\\sar_data\\part2_lookalike_mask.7z -oC:\\sar_data\\part2\\ -y')
    os.remove("C:/sar_data/part2_lookalike_mask.7z")
else:
    print("Mask_Lookalike уже есть на диске, скачивание пропускаем")


>>> wget -c -O C:/sar_data/part2_lookalike_mask.7z "https://zenodo.org/records/8253899/files/01_Train_Val_Lookalike_mask.7z?download=1"

--2026-05-30 02:26:46--  https://zenodo.org/records/8253899/files/01_Train_Val_Lookalike_mask.7z?download=1
Resolving zenodo.org (zenodo.org)... 188.184.103.118, 137.138.153.219, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|188.184.103.118|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 426795 (417K) [application/octet-stream]
Saving to: 'C:/sar_data/part2_lookalike_mask.7z'

     0K .......... .......... .......... .......... .......... 11%  331K 1s
    50K .......... .......... .......... .......... .......... 23% 1,58M 1s
   100K .......... .......... .......... .......... .......... 35% 1,20M 0s
   150K .......... .......... .......... .......... .......... 47% 27,8M 0s
   200K .......... .......... .......... .......... .......... 59% 5,13M 0s
   250K .......... .......... .......... .......... .......... 7

In [58]:
print("\nПроверка части 2:")
for folder in ["No_oil", "Lookalike", "Mask_no_oil", "Mask_lookalike"]:
    path = f"C:/sar_data/part2/{folder}"
    n = len([f for f in os.listdir(path) if f.endswith(".tif")]) if os.path.exists(path) else "Папка не найдена"
    print(f"  {folder}: {n} .tif файлов")


Проверка части 2:
  No_oil: 685 .tif файлов
  Lookalike: 685 .tif файлов
  Mask_no_oil: 685 .tif файлов
  Mask_lookalike: 685 .tif файлов


In [59]:
upload_all_folders(
    src_folders=[
        "C:/sar_data/part2/No_oil",
        "C:/sar_data/part2/Lookalike",
        "C:/sar_data/part2/Mask_no_oil",
        "C:/sar_data/part2/Mask_lookalike"
    ],
    slug="sar-oil-spill-part2",
    title="SAR Oil Spill Part2"
)


Копируем No_oil...
  Скопировано: 685 .tif файлов

Копируем Lookalike...
  Скопировано: 685 .tif файлов

Копируем Mask_no_oil...
  Скопировано: 685 .tif файлов

Копируем Mask_lookalike...
  Скопировано: 685 .tif файлов

Начинаем загрузку на Kaggle...

>>> kaggle datasets create -p C:/kaggle_staging --dir-mode tar
Starting upload for file Lookalike.tar
SOCKSHTTPSConnectionPool(host='www.googleapis.com', port=443): Max retries exceeded with url: /upload/storage/v1/b/kaggle-data-sets/o?uploadType=resumable&upload_id=AAVLpEhs91eVa8tBwWzHNh5MDhQFD8Vl_glRKD0PKzbunaZkv6rw2EUMsB2k-v8v-Xbd1GROqChXV-0J7MQI1A2QZ7e4Me1usBL97aMpU9YTLg (Caused by NewConnectionError('<urllib3.contrib.socks.SOCKSHTTPSConnection object at 0x00000133E8EBCCD0>: Failed to establish a new connection: [WinError 10061] Подключение не установлено, т.к. конечный компьютер отверг запрос на подключение'))
Upload successful: Lookalike.tar (27GB)
Starting upload for file Mask_lookalike.tar
Upload successful: Mask_lookalike.tar (3

## Часть 3 - тестовые снимки (Oil + No_Oil + Lookalike + маски): по 150 снимков + маски к ним

In [60]:
os.makedirs("C:/sar_data/part3", exist_ok=True)

In [61]:
if not os.path.exists("C:/sar_data/part3/Images"):
    run('wget -c -O C:/sar_data/part3.7z "https://zenodo.org/records/13761290/files/02_Test_images_and_ground_truth.7z?download=1"')
    run('"C:\\Program Files\\7-Zip\\7z.exe" x C:\\sar_data\\part3.7z -oC:\\sar_data\\part3\\ -y')
    os.remove("C:/sar_data/part3.7z")
else:
    print("Часть 3 уже есть на диске, скачивание пропускаем")


>>> wget -c -O C:/sar_data/part3.7z "https://zenodo.org/records/13761290/files/02_Test_images_and_ground_truth.7z?download=1"


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)




7-Zip 26.01 (x64) : Copyright (c) 1999-2026 Igor Pavlov : 2026-04-27

Scanning the drive for archives:
1 file, 9859650011 bytes (9403 MiB)

Extracting archive: C:\sar_data\part3.7z
--
Path = C:\sar_data\part3.7z
Type = 7z
Physical Size = 9859650011
Headers Size = 13416
Method = LZMA:23
Solid = +
Blocks = 1

Everything is Ok

Folders: 8
Files: 900
Size:       17000181900
Compressed: 9859650011




In [65]:
print("\nПроверка части 3:")
for folder in ["Images/Lookalike", "Images/No oil", "Images/Oil", "Mask/Lookalike", "Mask/No oil", "Mask/Oil"]:
    path = f"C:/sar_data/part3/{folder}"
    n = len([f for f in os.listdir(path) if f.endswith(".tif")]) if os.path.exists(path) else "Папка не найдена"
    print(f"  {folder}: {n} .tif файлов")


Проверка части 3:
  Images/Lookalike: 150 .tif файлов
  Images/No oil: 150 .tif файлов
  Images/Oil: 150 .tif файлов
  Mask/Lookalike: 150 .tif файлов
  Mask/No oil: 150 .tif файлов
  Mask/Oil: 150 .tif файлов


In [66]:
upload_all_folders(
    src_folders=[
        "C:/sar_data/part3/Images",
        "C:/sar_data/part3/Mask"
    ],
    slug="sar-oil-spill-part3",
    title="SAR Oil Spill Part3"
)


Копируем Images...
  Скопировано: 0 .tif файлов

Копируем Mask...
  Скопировано: 0 .tif файлов

Начинаем загрузку на Kaggle...

>>> kaggle datasets create -p C:/kaggle_staging --dir-mode tar
Dataset creation error: The requested title "SAR Oil Spill Part3" is already in use by a dataset. Please choose another title.



Готово: https://www.kaggle.com/datasets/daryanikitina/sar-oil-spill-part3
